# Interpretability & Code-Switching Analysis
Questo notebook analizza l'interpretabilità dei modelli per la **Dialect Identification** con particolare attenzione a:
- **Code-Switching (Italiano Standard $\leftrightarrow$ Dialetti)** sui post reali dei social media.
- **Token-Level Attribution tramite SHAP (Partition / Text Explainer)**: come il modello isola gli *shibboleths* dialettali all'interno della frase in italiano.
- **Ispezione Interna del Ramo Ausiliario (Feature Fusion)**: quali feature morfologiche/simboliche (n-grammi, prefissi/suffissi, fertility, marcatori regex) contribuiscono maggiormente alla classificazione.

## 1. Setup, Import e Caricamento dei Modelli Addestrati

In [ ]:
import sys
import os
sys.path.insert(0, 'src')

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer
from IPython.display import display, HTML

from dataset import DIALECT_CLASSES, LABEL2ID, ID2LABEL
from models import BaselineBERT, DialectFeatureFusionBERT
from explainability import ModelPredictWrapper, inspect_auxiliary_features, highlight_tokens_html

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device in uso per l'explainability: {device}")

model_name = 'models/bert-base-italian-xxl-cased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Caricamento Modello Baseline
model_base = BaselineBERT(model_name=model_name, num_classes=len(DIALECT_CLASSES))
model_base.load_state_dict(torch.load('checkpoints/best_baseline_model.pt', map_location=device))
model_base.to(device).eval()

# Caricamento Modello Feature Fusion ed Estrattore Ausiliario
extractor = joblib.load('checkpoints/fusion_feature_extractor.joblib')
aux_dim = extractor.n_features_
model_fusion = DialectFeatureFusionBERT(model_name=model_name, num_classes=len(DIALECT_CLASSES), aux_dim=aux_dim)
model_fusion.load_state_dict(torch.load('checkpoints/best_fusion_model.pt', map_location=device))
model_fusion.to(device).eval()

print("-> Tutti i modelli e gli estrattori sono stati caricati con successo!")

## 2. Selezione ed Esplorazione di Casi Reali di Code-Switching
Estraiamo esempi di tweet reali geolocalizzati da  contenenti commutazione di codice (inter- o intra-frasale).

In [ ]:
df_social = pd.read_csv('Social_media_posts/clean.csv')
print(f"Totale post social disponibili: {len(df_social):,}")

# Esempi selezionati ad alto tasso di code-switching (Campania/Napoli, Veneto, Sicilia, Lombardia)
sample_cases = [
    # Caso 1: Napoletano / Campano con matrice italiana
    "Andrea che non voleva alzarsi e fa teng na mutanda bianca",
    "Stamattina stavo andando a lavoro ma nun se po fa niente con sto traffico",
    # Caso 2: Veneto con espressioni e clitici
    "Tutti a parlare ma qua no se capise niente e no go mia tempo da perder",
    # Caso 3: Siciliano con shibboleth fonetico
    "Ma che cosa dici, chisto è veramente un beddu picciriddu",
    # Caso 4: Italiano standard di controllo
    "Oggi pomeriggio andiamo a fare una passeggiata in centro città con gli amici"
]

pred_wrapper_base = ModelPredictWrapper(model_base, tokenizer, device, is_fusion=False)
pred_wrapper_fusion = ModelPredictWrapper(model_fusion, tokenizer, device, is_fusion=True, extractor=extractor)

print("=== PREDIZIONI SUI CASI DI CODE-SWITCHING ===")
probs_base = pred_wrapper_base(sample_cases)
probs_fusion = pred_wrapper_fusion(sample_cases)

for i, text in enumerate(sample_cases):
    pred_b = ID2LABEL[np.argmax(probs_base[i])]
    conf_b = np.max(probs_base[i]) * 100
    pred_f = ID2LABEL[np.argmax(probs_fusion[i])]
    conf_f = np.max(probs_fusion[i]) * 100
    print(f'\n[{i+1}] Testo: "{text}"')
    print(f"    -> Baseline BERT:       {pred_b} ({conf_b:.1f}%)")
    print(f"    -> Feature Fusion BERT: {pred_f} ({conf_f:.1f}%)")

## 3. Ispezione Interna del Ramo Ausiliario (Feature Fusion)
Per comprendere perché il modello Feature Fusion prende una specifica decisione in presenza di code-switching, ispezioniamo quali delle **938 feature simboliche** si sono attivate con il massimo impatto proiettato.

In [ ]:
target_sample = sample_cases[1] # "Stamattina stavo andando a lavoro ma nun se po fa niente con sto traffico"
print(f'Analisi feature ausiliarie per: "{target_sample}"\n')

df_feats = inspect_auxiliary_features(model_fusion, extractor, target_sample, top_k=15)
display(df_feats)

# Visualizzazione a barre delle Top-15 Feature
plt.figure(figsize=(10, 6))
colors = ['#27ae60' if x > 0 else '#e74c3c' for x in df_feats['Impatto Proiettato']]
plt.barh(df_feats['Feature'][::-1], df_feats['Impatto Proiettato'][::-1], color=colors[::-1], edgecolor='black', alpha=0.85)
plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
plt.title(f'Top-15 Feature Simboliche nel Ramo Ausiliario\n("{target_sample[:50]}...")', fontsize=12, fontweight='bold')
plt.xlabel("Impatto Proiettato nel Layer di Fusione", fontsize=10)
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 4. SHAP Partition Explainer (Text Masking)
Utilizziamo **SHAP** ( con ) per calcolare i valori di Shapley a livello di parola e token.
Questo metodo isola formalmente il contributo marginale di ogni singolo termine rispetto alla predizione finale.

In [ ]:
import shap

# Inizializziamo il masker di testo basato sul tokenizer di BERT
masker = shap.maskers.Text(tokenizer)

# Creiamo l'explainer SHAP sul wrapper del modello proposto (Feature Fusion)
explainer_fusion = shap.Explainer(pred_wrapper_fusion, masker, output_names=DIALECT_CLASSES)

# Calcoliamo i valori SHAP sui casi di code-switching
print("Calcolo valori SHAP in corso su GPU...")
shap_values = explainer_fusion(sample_cases)
print("-> SHAP values calcolati!")

# Esempio: Visualizzazione Waterfall / Bar Plot sul primo caso di Code-Switching
pred_class_idx = np.argmax(probs_fusion[0])
pred_class_name = ID2LABEL[pred_class_idx]
print(f"Visualizzazione SHAP per la classe predetta: {pred_class_name}")
shap.plots.text(shap_values[0, :, pred_class_idx])

## 5. Confronto di Spiegabilità: Baseline BERT vs Feature Fusion sul Code-Switching
Confrontiamo i grafici SHAP di entrambi i modelli sulla stessa frase per verificare se l'integrazione neuro-simbolica aumenta l'interpretabilità e riduce i falsi positivi.

In [ ]:
explainer_base = shap.Explainer(pred_wrapper_base, masker, output_names=DIALECT_CLASSES)
shap_values_base = explainer_base([sample_cases[1]])
shap_values_fusion = explainer_fusion([sample_cases[1]])

nap_idx = LABEL2ID['nap']
print("--- SHAP ATTRIBUTION: BASELINE BERT (Puro) ---")
display(shap.plots.text(shap_values_base[0, :, nap_idx]))

print("--- SHAP ATTRIBUTION: FEATURE FUSION BERT (Proposto) ---")
display(shap.plots.text(shap_values_fusion[0, :, nap_idx]))